# GRACE H3-v2 — Bắt đầu full pipeline tại đây

Branch: **feat/h3-v2-benchmark**. MCC là chính, F1 thứ hai; chi phí phụ. Cấu hình mặc định: Devign + ReVeal, seed42. Không có điều kiện bỏ sót1% trong nhánh benchmark.

| Lần chạy local | Thực hiện | Chuyển môi trường |
|---|---|---|
| 1 | Tạo/kiểm tra CPU input | **DỪNG → Colab, PHASE='base'** |
| 2 | Nhập base-output; preflight và gọi GRACE validation | **DỪNG → Colab, PHASE='meta'** |
| 3 | Nhập meta-output; GRACE test; predict/evaluate | Đọc bảng benchmark |

Sau mỗi lần lấy output từ Colab, đặt zip vào bundles/ rồi Run All lại. Các stage hoàn chỉnh và response exact-cache được reuse. Notebook Colab: **colab_cpu_prepare.ipynb**, accelerator **None**. GPU/Kaggle chưa cần cho mặc định.

Validation có tối đa4.920 query trước dedup/cache; test có5.006 query cho hai dataset. Preflight báo số request thực cần tạo. RUN_INFERENCE=True cho phép gọi LLM khi bạn chạy notebook; False chỉ dùng cache.

Đây là implementation mới chưa có benchmark H3-v2. Test đã được xem ở các vòng trước; không dùng test để chọn winner.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess
candidates=[Path.cwd(),Path.cwd().parent,Path('D:/JUNIOR/LabThayTho/GRACE-H3-BenchmarkEnsemble'),Path('/mnt/d/JUNIOR/LabThayTho/GRACE-H3-BenchmarkEnsemble')]
if os.environ.get('GRACE_H3V2_ROOT'): candidates.insert(0,Path(os.environ['GRACE_H3V2_ROOT']))
ROOT=next((p.resolve() for p in candidates if (p/'configs/benchmark_v2.json').exists()),None)
if ROOT is None: raise RuntimeError('Mở notebook từ worktree GRACE-H3-BenchmarkEnsemble hoặc đặt GRACE_H3V2_ROOT')
sys.path.insert(0,str(ROOT))
INSTALL_DEPENDENCIES=True
PREPARE_ON_COLAB=True
RUN_INFERENCE=True
BUNDLES=ROOT/'bundles'
BUNDLES.mkdir(exist_ok=True)
print('ROOT:',ROOT)
print('Entry của branch benchmark; hai notebook H3 cũ ở worktree khác vẫn giữ nguyên.')


In [ ]:
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable,'-m','pip','install','--break-system-packages','-r',str(ROOT/'requirements-v2.txt'),'-r',str(ROOT/'requirements.txt')])
from grace_h1.io import load_local_environment
load_local_environment(ROOT)
from grace_h3v2 import workflow as w
from grace_h3v2.portable import create_bundle,import_bundle,identity
import zipfile
import pandas as pd
CFG=w.config(ROOT)
print(json.dumps(CFG,indent=2))
def show_stage(name):
    try: return w.load_stage(ROOT,CFG,name)
    except FileNotFoundError: return None
def cpu_handoff(phase,selection):
    output=BUNDLES/f'grace-h3v2-{phase}-output.zip'
    if output.exists(): print(import_bundle(ROOT,output,'output',phase))
    result=show_stage(selection)
    if result is not None: return result
    if not PREPARE_ON_COLAB:
        return w.base_prepare(ROOT,CFG) if phase=='base' else w.meta_prepare(ROOT,CFG)
    archive=BUNDLES/f'grace-h3v2-{phase}-inputs.zip'
    reuse=False
    if archive.exists():
        with zipfile.ZipFile(archive) as z:
            m=json.loads(z.read('bundle_manifest.json'))
            reuse=m.get('job')==identity(ROOT) and m.get('phase')==phase and m.get('kind')=='inputs'
    if not reuse: print(create_bundle(ROOT,archive,'inputs',phase))
    raise RuntimeError(f'DỪNG: mở colab_cpu_prepare.ipynb trên Colab CPU, đặt PHASE={phase!r}; upload {archive.name} và grace-h3-joern-runtime.zip. Lấy {output.name} về bundles/ rồi Run All lại.')


## Bước 1 — Colab CPU: classifier và chuẩn bị validation

Lần đầu sẽ dừng tại cell tiếp theo. Colab PHASE='base' fit classifier bằng nested train OOF, chọn gate bằng MCC/F1, chuẩn bị prompts validation và trả base-output. CodeT5 frozen; không gọi LLM trên Colab.


In [ ]:
base=w.base_prepare(ROOT,CFG) if not PREPARE_ON_COLAB else cpu_handoff('base','base_selection')
print(pd.DataFrame([{'dataset':j['dataset'],'seed':j['seed'],'base_folder':j['base_folder']} for j in base['jobs']]))


## Bước 2 — Local: GRACE validation ngoài test

Dùng ADC/Vertex của môi trường local hiện có. Preflight trước, rồi chỉ tạo missing exact requests. Validation labels dùng học meta-model sau đó; test responses không dùng huấn luyện. Run All lại sẽ giữ inference run đã complete, không ghi lại prediction metadata.


In [ ]:
plans=w.preflight(ROOT,CFG,'validation')
print(json.dumps(plans,indent=2))
reports=w.infer(ROOT,CFG,'validation',allow_api=RUN_INFERENCE)
print(json.dumps(reports,indent=2))
if any(r['status']!='complete' for r in reports): raise RuntimeError('Validation chưa complete: bật RUN_INFERENCE hoặc xử lý API/ADC rồi chạy lại. Chưa chuyển meta-fit.')


## Bước 3 — Colab CPU lần hai: router/stacking và test features

Cell tiếp theo tạo meta-inputs rồi dừng nếu chưa có meta-output. Mở cùng notebook Colab, đổi **PHASE='meta'**, upload meta-inputs và runtime. Colab fit meta OOF trên eligible validation, khóa winner MCC/F1, refit, tính test features không nhãn và chuẩn bị full-test GRACE.

Nhánh benefit-router chỉ dùng feature trước expert; selector/stacking dùng expert output. Full-reference/stacking có thể cần100% query: đây là benchmark mode.


In [ ]:
meta=w.meta_prepare(ROOT,CFG) if not PREPARE_ON_COLAB else cpu_handoff('meta','meta_selection')
print(pd.DataFrame([{'dataset':j['dataset'],'seed':j['seed'],'winner_development':j['development_winner']} for j in meta['jobs']]))


## Bước 4 — Local: test inference và đánh giá

Winner đã khóa từ development. Preflight/test-infer reuse response nếu prompt/model/decoding giống hệt. Local chỉ kết hợp các score nhỏ đã tính ở Colab. Nhãn test chỉ mở trong evaluate; không chọn winner theo bảng test.


In [ ]:
print(json.dumps(w.preflight(ROOT,CFG,'test'),indent=2))
reports=w.infer(ROOT,CFG,'test',allow_api=RUN_INFERENCE)
print(json.dumps(reports,indent=2))
if any(r['status']!='complete' for r in reports): raise RuntimeError('Test requests chưa complete; resume inference trước evaluate.')
w.predict(ROOT,CFG)
summary=w.evaluate(ROOT,CFG)


In [ ]:
table=[]
for run in summary['datasets']:
    for arm,result in run['arms'].items():
        m=result['metrics']
        table.append({'dataset':run['dataset'],'seed':run['seed'],'arm':arm,'locked_winner':arm==run['locked_development_winner'],'MCC':m['mcc'],'F1':m['f1'],'precision':m['precision'],'recall':m['recall'],'specificity':m['specificity'],'invalid':m['invalid_n'],'expert_query_ratio':result['algorithmic_expert_query_ratio'],'fallback':result['fallback_n']})
display(pd.DataFrame(table))
print('Summary:',w.stage_file(ROOT,CFG,'benchmark_summary'))
print('Không chọn lại winner dựa trên bảng test. So với gate_tuned và full_grace; chi phí routing lý thuyết tách actual API/cache.')
